In [ ]:
import pandas as pd
from multicall import Call
import numpy as np
from web3 import Web3


from mainnet_launch.database.schema.full import (
    DestinationStates,
    DestinationTokenValues,
    AutopoolDestinationStates,
    Autopools,
    DestinationTokens,
    Destinations,
    Tokens,
    TokenValues,
)


from mainnet_launch.abis import STATS_CALCULATOR_REGISTRY_ABI
from mainnet_launch.data_fetching.get_events import fetch_events


from mainnet_launch.database.schema.postgres_operations import (
    get_full_table_as_orm,
    insert_avoid_conflicts,
    get_subset_not_already_in_column,
    generic_merge_tables_as_df,
    get_full_table_as_df,
    natural_left_right_using_where,
)
from mainnet_launch.data_fetching.get_state_by_block import (
    get_raw_state_by_blocks,
    safe_normalize_with_bool_success,
    build_blocks_to_use,
    identity_with_bool_success,
    get_state_by_one_block,
)
from mainnet_launch.constants import ALL_CHAINS, ROOT_PRICE_ORACLE, ChainData, STATS_CALCULATOR_REGISTRY, WETH


chain = ALL_CHAINS[0]
possible_blocks = build_blocks_to_use(chain)

missing_blocks = get_subset_not_already_in_column(
    DestinationTokenValues,
    DestinationTokenValues.block,
    possible_blocks,
    where_clause=DestinationTokenValues.chain_id == chain.chain_id,
)

full_destination_df = generic_merge_tables_as_df(
    DestinationTokens,
    Destinations,
    left_join_columns=[DestinationTokens.destination_vault_address, DestinationTokens.chain_id],
    right_join_columns=[Destinations.destination_vault_address, Destinations.chain_id],
    where_clause=None,
)


# def

full_destination_df.columns
#     DestinationTokens, where_clause=DestinationTokens.chain_id == chain.chain_id
# )
# all_destinations: list[Destinations] = get_full_table_as_orm(
#     Destinations, where_clause=Destinations.chain_id == chain.chain_id
# )

2025-04-25 11:52:48,324 INFO sqlalchemy.engine.Engine select pg_catalog.version()
2025-04-25 11:52:48,325 INFO sqlalchemy.engine.Engine [raw sql] {}
2025-04-25 11:52:48,525 INFO sqlalchemy.engine.Engine select current_schema()
2025-04-25 11:52:48,526 INFO sqlalchemy.engine.Engine [raw sql] {}
2025-04-25 11:52:48,719 INFO sqlalchemy.engine.Engine show standard_conforming_strings
2025-04-25 11:52:48,722 INFO sqlalchemy.engine.Engine [raw sql] {}
2025-04-25 11:52:48,941 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-04-25 11:52:48,948 INFO sqlalchemy.engine.Engine SELECT max(blocks.block) AS block 
FROM blocks 
WHERE blocks.chain_id = %(chain_id_1)s AND blocks.block >= %(block_1)s AND blocks.block <= %(block_2)s GROUP BY date_trunc(%(date_trunc_1)s, blocks.datetime) ORDER BY date_trunc(%(date_trunc_2)s, blocks.datetime)
2025-04-25 11:52:48,949 INFO sqlalchemy.engine.Engine [generated in 0.00042s] {'chain_id_1': 1, 'block_1': 20752910, 'block_2': 100000000, 'date_trunc_1': 'day', 'dat

Index(['destination_vault_address', 'chain_id', 'token_address', 'index',
       'destination_vault_address', 'chain_id', 'exchange_name',
       'block_deployed', 'name', 'symbol', 'pool_type', 'pool', 'underlying',
       'underlying_symbol', 'underlying_name'],
      dtype='object')

In [2]:
def build_pool_token_spot_price_calls(
    chain: ChainData, pool_addresses: list[str], token_addresses: list[str]
) -> list[Call]:

    return [
        Call(
            ROOT_PRICE_ORACLE(chain),
            ["getSpotPriceInEth(address,address)(uint256)", token_address, pool_address],
            [(f"{pool_address}_{token_address}", safe_normalize_with_bool_success)],
        )
        for (pool_address, token_address) in zip(pool_addresses, token_addresses)
    ]


spot_price_calls = build_pool_token_spot_price_calls(
    chain, full_destination_df["pool"], full_destination_df["token_address"]
)
destination_token_spot_price_df = get_raw_state_by_blocks(
    spot_price_calls, possible_blocks, chain, include_block_number=True
)
destination_token_spot_price_df  # each of these cells is a DestinationTokenValues() less quantity

,0x1E19CF2D73a72Ef1332C882F20534B6519Be0276_0xae78736Cd615f374D3085123A210448E74Fc6393,0x1E19CF2D73a72Ef1332C882F20534B6519Be0276_0xC02aaA39b223FE8D0A0e5C4F27eAD9083C756Cc2,0xDACf5Fa19b1f720111609043ac67A9818262850c_0xC02aaA39b223FE8D0A0e5C4F27eAD9083C756Cc2,0xDACf5Fa19b1f720111609043ac67A9818262850c_0xf1C9acDc66974dFB6dEcB12aA385b9cD01190E38,0x93d199263632a4EF4Bb438F1feB99e57b4b5f0BD_0x7f39C581F595B53c5cb19bD0b3f8dA6c935E2Ca0,0x93d199263632a4EF4Bb438F1feB99e57b4b5f0BD_0xC02aaA39b223FE8D0A0e5C4F27eAD9083C756Cc2,0xf01b0684C98CD7aDA480BFDF6e43876422fa1Fc1_0x7f39C581F595B53c5cb19bD0b3f8dA6c935E2Ca0,0xf01b0684C98CD7aDA480BFDF6e43876422fa1Fc1_0xC02aaA39b223FE8D0A0e5C4F27eAD9083C756Cc2,0xF7A826D47c8E02835D94fb0Aa40F0cC9505cb134_0x7f39C581F595B53c5cb19bD0b3f8dA6c935E2Ca0,0xF7A826D47c8E02835D94fb0Aa40F0cC9505cb134_0xBe9895146f7AF43049ca1c1AE358B0541Ea49704,...,0x91F0f34916Ca4E2cCe120116774b0e4fA0cdcaA8_0x4200000000000000000000000000000000000006,0xA6385c73961dd9C58db2EF0c4EB98cE4B60651e8_0x4200000000000000000000000000000000000006,0xA6385c73961dd9C58db2EF0c4EB98cE4B60651e8_0xc1CBa3fCea344f92D9239c08C0568f6F2F0ee452,0xA24382874A6FD59de45BbccFa160488647514c28_0x4200000000000000000000000000000000000006,0xA24382874A6FD59de45BbccFa160488647514c28_0xEDfa23602D0EC14714057867A78d01e94176BEA0,0x44Ecc644449fC3a9858d2007CaA8CFAa4C561f91_0x2Ae3F1Ec7F1F5012CFEab0185bfc7aa3cf0DEc22,0x44Ecc644449fC3a9858d2007CaA8CFAa4C561f91_0x4200000000000000000000000000000000000006,0x54D86E177cdC664B5F9B17eB5fd6A76Fa529e466_0x2Ae3F1Ec7F1F5012CFEab0185bfc7aa3cf0DEc22,0x54D86E177cdC664B5F9B17eB5fd6A76Fa529e466_0xc1CBa3fCea344f92D9239c08C0568f6F2F0ee452,block
timestamp,,,,,,,,,,,,,,,,,,,,,
2024-09-15 23:59:59+00:00,1.117733,0.999825,0.999958,1.028677,1.178367,1.000137,1.178423,1.000090,1.179269,1.078810,...,None,None,None,None,None,None,None,None,None,20759464
2024-09-16 23:59:59+00:00,1.117901,0.998980,0.999854,1.028862,1.178241,1.000212,1.178180,1.000264,1.178708,1.079182,...,None,None,None,None,None,None,None,None,None,20766617
2024-09-17 23:59:59+00:00,1.117654,0.999802,0.999957,1.028940,1.177920,1.000300,1.177957,1.000269,1.177578,1.079637,...,None,None,None,None,None,None,None,None,None,20773761
2024-09-18 23:59:59+00:00,1.118010,0.999633,0.999909,1.029087,1.178502,1.000268,1.178266,1.000468,1.179518,1.079779,...,None,None,None,None,None,None,None,None,None,20780916
2024-09-19 23:59:59+00:00,1.117595,1.000023,1.000051,1.029067,1.178894,0.999633,1.178926,0.999606,1.178245,1.079739,...,None,None,None,None,None,None,None,None,None,20788084
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-04-20 23:59:59+00:00,1.126730,0.999578,0.999957,1.044191,1.199472,1.000046,1.199565,0.999968,1.199766,1.095072,...,None,None,None,None,None,None,None,None,None,22313650
2025-04-21 23:59:47+00:00,1.126735,0.999699,1.000018,1.044249,1.199729,0.999940,1.199655,1.000002,1.200538,1.095277,...,None,None,None,None,None,None,None,None,None,22320818
2025-04-22 23:59:59+00:00,1.126670,0.999848,0.999993,1.044353,1.199610,1.000114,1.199757,0.999991,1.200260,1.095364,...,None,None,None,None,None,None,None,None,None,22327989


In [3]:
def build_reserve_values_in_destinations():
    

SyntaxError: incomplete input (3340202141.py, line 2)

In [ ]:
full_destination_df

,destination_vault_address,chain_id,token_address,index,destination_vault_address,chain_id,exchange_name,block_deployed,name,symbol,pool_type,pool,underlying,underlying_symbol,underlying_name
0,0xf3ae3c74EaD129e770A864CeE291A805b170bBe0,1,0xae78736Cd615f374D3085123A210448E74Fc6393,0,0xf3ae3c74EaD129e770A864CeE291A805b170bBe0,1,balancer,20756405,Tokemak-Wrapped Ether-Balancer rETH Stable Pool,toke-WETH-B-rETH-STABLE,balMetaStable,0x1E19CF2D73a72Ef1332C882F20534B6519Be0276,0x1E19CF2D73a72Ef1332C882F20534B6519Be0276,B-rETH-STABLE,B-rETH-STABLE
1,0xf3ae3c74EaD129e770A864CeE291A805b170bBe0,1,0xC02aaA39b223FE8D0A0e5C4F27eAD9083C756Cc2,1,0xf3ae3c74EaD129e770A864CeE291A805b170bBe0,1,balancer,20756405,Tokemak-Wrapped Ether-Balancer rETH Stable Pool,toke-WETH-B-rETH-STABLE,balMetaStable,0x1E19CF2D73a72Ef1332C882F20534B6519Be0276,0x1E19CF2D73a72Ef1332C882F20534B6519Be0276,B-rETH-STABLE,B-rETH-STABLE
2,0x865e59D439BF7310c9BC6117E6020B8C87De4065,1,0xC02aaA39b223FE8D0A0e5C4F27eAD9083C756Cc2,0,0x865e59D439BF7310c9BC6117E6020B8C87De4065,1,balancer,20756406,Tokemak-Wrapped Ether-Balancer osETH/wETH Stab...,toke-WETH-osETH/wETH-BPT,balCompStable,0xDACf5Fa19b1f720111609043ac67A9818262850c,0xDACf5Fa19b1f720111609043ac67A9818262850c,osETH/wETH-BPT,osETH/wETH-BPT
3,0x865e59D439BF7310c9BC6117E6020B8C87De4065,1,0xf1C9acDc66974dFB6dEcB12aA385b9cD01190E38,1,0x865e59D439BF7310c9BC6117E6020B8C87De4065,1,balancer,20756406,Tokemak-Wrapped Ether-Balancer osETH/wETH Stab...,toke-WETH-osETH/wETH-BPT,balCompStable,0xDACf5Fa19b1f720111609043ac67A9818262850c,0xDACf5Fa19b1f720111609043ac67A9818262850c,osETH/wETH-BPT,osETH/wETH-BPT
4,0x25cb41919d6B88e0D48108A4F5fe8FBb3744aFE1,1,0x7f39C581F595B53c5cb19bD0b3f8dA6c935E2Ca0,0,0x25cb41919d6B88e0D48108A4F5fe8FBb3744aFE1,1,balancer,20756407,Tokemak-Wrapped Ether-Balancer wstETH-WETH Sta...,toke-WETH-wstETH-WETH-BPT,balCompStable,0x93d199263632a4EF4Bb438F1feB99e57b4b5f0BD,0x93d199263632a4EF4Bb438F1feB99e57b4b5f0BD,wstETH-WETH-BPT,wstETH-WETH-BPT
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
165,0xA94031Ed4b316B043464FDd5482877F42A39845a,8453,0xEDfa23602D0EC14714057867A78d01e94176BEA0,1,0xA94031Ed4b316B043464FDd5482877F42A39845a,8453,aerodrome,22326389,Tokemak-Wrapped Ether-Volatile AMM - WETH/wrsETH,toke-WETH-vAMM-WETH/wrsETH,vAMM,0xA24382874A6FD59de45BbccFa160488647514c28,0xA24382874A6FD59de45BbccFa160488647514c28,vAMM-WETH/wrsETH,vAMM-WETH/wrsETH
166,0xd18db4dD6aF6A7536aD7F863C136463681e0CdAD,8453,0x2Ae3F1Ec7F1F5012CFEab0185bfc7aa3cf0DEc22,0,0xd18db4dD6aF6A7536aD7F863C136463681e0CdAD,8453,aerodrome,22326389,Tokemak-Wrapped Ether-Volatile AMM - cbETH/WETH,toke-WETH-vAMM-cbETH/WETH,vAMM,0x44Ecc644449fC3a9858d2007CaA8CFAa4C561f91,0x44Ecc644449fC3a9858d2007CaA8CFAa4C561f91,vAMM-cbETH/WETH,vAMM-cbETH/WETH
167,0xd18db4dD6aF6A7536aD7F863C136463681e0CdAD,8453,0x4200000000000000000000000000000000000006,1,0xd18db4dD6aF6A7536aD7F863C136463681e0CdAD,8453,aerodrome,22326389,Tokemak-Wrapped Ether-Volatile AMM - cbETH/WETH,toke-WETH-vAMM-cbETH/WETH,vAMM,0x44Ecc644449fC3a9858d2007CaA8CFAa4C561f91,0x44Ecc644449fC3a9858d2007CaA8CFAa4C561f91,vAMM-cbETH/WETH,vAMM-cbETH/WETH
168,0xBd137c56f3116E5c36753037a784FF844F84F59c,8453,0x2Ae3F1Ec7F1F5012CFEab0185bfc7aa3cf0DEc22,0,0xBd137c56f3116E5c36753037a784FF844F84F59c,8453,balancer,22661893,Tokemak-Wrapped Ether-Gyroscope ECLP cbETH/wstETH,toke-WETH-ECLP-cbETH-wstETH,balGyro,0x54D86E177cdC664B5F9B17eB5fd6A76Fa529e466,0x54D86E177cdC664B5F9B17eB5fd6A76Fa529e466,ECLP-cbETH-wstETH,ECLP-cbETH-wstETH


In [ ]:
full_destination_df["destination_vault_address"]

,destination_vault_address,destination_vault_address
0,0xf3ae3c74EaD129e770A864CeE291A805b170bBe0,0xf3ae3c74EaD129e770A864CeE291A805b170bBe0
1,0xf3ae3c74EaD129e770A864CeE291A805b170bBe0,0xf3ae3c74EaD129e770A864CeE291A805b170bBe0
2,0x865e59D439BF7310c9BC6117E6020B8C87De4065,0x865e59D439BF7310c9BC6117E6020B8C87De4065
3,0x865e59D439BF7310c9BC6117E6020B8C87De4065,0x865e59D439BF7310c9BC6117E6020B8C87De4065
4,0x25cb41919d6B88e0D48108A4F5fe8FBb3744aFE1,0x25cb41919d6B88e0D48108A4F5fe8FBb3744aFE1
...,...,...
165,0xA94031Ed4b316B043464FDd5482877F42A39845a,0xA94031Ed4b316B043464FDd5482877F42A39845a
166,0xd18db4dD6aF6A7536aD7F863C136463681e0CdAD,0xd18db4dD6aF6A7536aD7F863C136463681e0CdAD
167,0xd18db4dD6aF6A7536aD7F863C136463681e0CdAD,0xd18db4dD6aF6A7536aD7F863C136463681e0CdAD
168,0xBd137c56f3116E5c36753037a784FF844F84F59c,0xBd137c56f3116E5c36753037a784FF844F84F59c


In [ ]:
def build_underlying_reserves_calls(chain: ChainData, destinations: list[str]) -> list[Call]:
    return [
        Call(
            dest,
            ["underlyingReserves()(address[],uint256[])"],
            [
                (f"{dest}_underlyingReserves_tokens", identity_with_bool_success),
                (f"{dest}_underlyingReserves_amounts", identity_with_bool_success),
            ],
        )
        for dest in destinations
    ]

In [ ]:
underlying_reserves_call = Call(
    dest.vaultAddress,
    ["underlyingReserves()(address[],uint256[])"],
    [
        (f"{unique_id}_underlyingReserves_tokens", None),
        (f"{unique_id}_underlyingReserves_amounts", same_normalize_with_bool_success_list),
    ],
)

In [ ]:
token_value_df = get_full_table_as_df(TokenValues, where_clause=TokenValues.chain_id == chain.chain_id)
# (token_address, chain_id, block) -> [safe_price, backing]
token_value_df

2025-04-25 11:44:50,683 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-04-25 11:44:50,684 INFO sqlalchemy.engine.Engine SELECT pg_catalog.pg_class.relname 
FROM pg_catalog.pg_class JOIN pg_catalog.pg_namespace ON pg_catalog.pg_namespace.oid = pg_catalog.pg_class.relnamespace 
WHERE pg_catalog.pg_class.relname = %(table_name)s AND pg_catalog.pg_class.relkind = ANY (ARRAY[%(param_1)s, %(param_2)s, %(param_3)s, %(param_4)s, %(param_5)s]) AND pg_catalog.pg_table_is_visible(pg_catalog.pg_class.oid) AND pg_catalog.pg_namespace.nspname != %(nspname_1)s
2025-04-25 11:44:50,685 INFO sqlalchemy.engine.Engine [cached since 844.6s ago] {'table_name': <sqlalchemy.sql.elements.TextClause object at 0x13acf70a0>, 'param_1': 'r', 'param_2': 'p', 'param_3': 'f', 'param_4': 'v', 'param_5': 'm', 'nspname_1': 'pg_catalog'}
2025-04-25 11:44:50,686 INFO sqlalchemy.engine.Engine 
            SELECT *
            FROM token_values
            WHERE token_values.chain_id = 1
        
2025-04-25 11:44:50,68

,block,chain_id,token_address,denomiated_in,backing,safe_price
0,20759464,1,0xCAcd6fd266aF91b8AeD52aCCc382b4e165586E29,0xC02aaA39b223FE8D0A0e5C4F27eAD9083C756Cc2,NaN,NaN
1,20759464,1,0xdAC17F958D2ee523a2206206994597C13D831ec7,0xC02aaA39b223FE8D0A0e5C4F27eAD9083C756Cc2,NaN,NaN
2,20759464,1,0xC71Ea051a5F82c67ADcF634c36FFE6334793D24C,0xC02aaA39b223FE8D0A0e5C4F27eAD9083C756Cc2,NaN,NaN
3,20759464,1,0x5E8422345238F34275888049021821E8E08CAa1f,0xC02aaA39b223FE8D0A0e5C4F27eAD9083C756Cc2,NaN,0.999145
4,20759464,1,0xCd5fE23C85820F7B72D0926FC9b05b43E359b7ee,0xC02aaA39b223FE8D0A0e5C4F27eAD9083C756Cc2,NaN,1.047726
...,...,...,...,...,...,...
6433,22342307,1,0xf939E0A03FB07F59A73314E73794Be0E57ac1b4E,0xC02aaA39b223FE8D0A0e5C4F27eAD9083C756Cc2,NaN,0.000566
6434,22342307,1,0x83F20F44975D03b1b09e64809B757c47f942BEeA,0xC02aaA39b223FE8D0A0e5C4F27eAD9083C756Cc2,NaN,0.000654
6435,22342307,1,0xA0b86991c6218b36c1d19D4a2e9Eb0cE3606eB48,0xC02aaA39b223FE8D0A0e5C4F27eAD9083C756Cc2,NaN,0.000566
6436,22342307,1,0xA1290d69c65A6Fe4DF752f95823fae25cB99e5A7,0xC02aaA39b223FE8D0A0e5C4F27eAD9083C756Cc2,NaN,1.041302


In [ ]:
a = DestinationTokenValues()

TypeError: __init__() missing 8 required positional arguments: 'block', 'chain_id', 'token_address', 'destination_address', 'spot_price', 'quantity', 'safe_spot_spread', and 'spot_backing_discount'